# Precision & Recall: LLM Content Moderation

This notebook walks through how to evaluate an LLM-based classifier using **precision** and **recall** — two of the most important metrics for content moderation and classification tasks.

## The Dataset
We're working with app store reviews that have been **manually labeled** by humans for three types of harmful content:
- `racism`
- `bullying`  
- `sexual`

Reviews that have been labeled (value is `0` or `1`) form the **golden set** — our ground truth.

## The Goal
We'll build an LLM-based classifier that predicts whether a review contains unwanted sexual content, then measure how well it performs against the human labels.

---

## Key Concepts

**Precision** answers: *"Of everything the model flagged, what fraction was actually harmful?"*
- High precision = few false alarms
- `precision = true_positives / (true_positives + false_positives)`

**Recall** answers: *"Of all the actually harmful content, what fraction did the model catch?"*
- High recall = few misses
- `recall = true_positives / (true_positives + false_negatives)`

There's usually a trade-off: tuning your prompt to catch more bad content (higher recall) tends to also flag more innocent content (lower precision), and vice versa. Understanding this trade-off is the whole point of this exercise.


---
## Step 1: Read the Data

In [11]:
import pandas as pd

df = pd.read_csv("reviews-marked.csv")

print(f"Total rows: {len(df)}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(10)


Total rows: 56056

Column names: ['Country', 'Date', 'Rating', 'Review', 'Version', 'source', 'racism', 'bullying', 'sexual']

First few rows:


,Country,Date,Rating,Review,Version,source,racism,bullying,sexual
0,US,11/22/2019,5,It’s a great app to meet new people and chat in very satisfied with downloading this app i recommend this app if you like to chat or just to meet new people. And you can choose which country To find different users!,4.4.5,holla,NaN,NaN,NaN
1,US,11/22/2019,5,"Holla is an excellent app, where I get to know new people every time and even get to make new friends. I truly recommend this application to all people!",4.4.5,holla,NaN,NaN,NaN
2,US,11/22/2019,1,Get rid of micro transactions or i will find a new app to use. Why should i have to pay for that it’s so stupid,3.8,holla,0.0,0.0,0.0
3,US,11/22/2019,5,"Free to use app, meet people around the world.",-,holla,NaN,NaN,NaN
4,US,11/21/2019,5,I got this app and everything has been different. I’ve met so many interesting people. From around the world. I was recently reunited with my high school girlfriend. We’re getting married. I met and married The love of my Life thanks to Holla. Thanks Holla!!!!!,4.4.5,holla,NaN,NaN,NaN
5,US,11/21/2019,5,You make good friends here,4.4.5,holla,NaN,NaN,NaN
6,US,11/21/2019,1,This is good but most of my messages never show up. This is very crapy and needs to be fixed,3.3.1,skout,0.0,0.0,0.0
7,US,11/20/2019,4,App is ok only thing is that people can’t here or see me sometimes don’t know if it’s in my end or theres,4.4.5,holla,NaN,NaN,NaN
8,US,11/20/2019,1,I was really enjoying this app. This brought me out of the box. I’m an extremely shy person and this gave me somewhere to talk to nice people. I just got kicked of bc I’m 16 not “18” and I think that this change it kind of stupid bc yeah it’s for protection but like someone else said all you hav...,4.2.1,holla,0.0,0.0,0.0
9,US,11/18/2019,5,Good,4.4.4,holla,NaN,NaN,NaN


---
## Step 2: Filter to the Golden Set

The **golden set** is the subset of reviews that have been manually labeled by a human. You can identify them because the `racism`, `bullying`, and `sexual` columns have actual values (`0` or `1`) rather than being empty.

These are the only rows we can use to evaluate our classifier — we need ground truth labels to calculate precision and recall.

> **Note:** A value of `1` means the review *contains* that type of content (e.g., unwanted sexual comments). A value of `0` means a human reviewed it and decided it does *not*. Empty/NaN means the review was never labeled for that category — which is **different** from `0`.

Since we're building a classifier for sexual content, we filter down further to only rows where the `sexual` column was actually labeled.


In [12]:
# A row is in the golden set if ANY of the three label columns has a value.
# notna() returns True if the value is not NaN (i.e., the cell has been filled in).
golden = df[
    df["racism"].notna() | df["bullying"].notna() | df["sexual"].notna()
].copy()

print(f"Total rows in dataset:  {len(df)}")
print(f"Rows in golden set:     {len(golden)}")
print(f"\nHow many have each label filled in:")
print(f"  racism labeled:   {golden['racism'].notna().sum()}")
print(f"  bullying labeled: {golden['bullying'].notna().sum()}")
print(f"  sexual labeled:   {golden['sexual'].notna().sum()}")

# For our sexual content evaluation, we only want rows where sexual was actually labeled.
# NaN means nobody categorized it — it's NOT the same as 0 (human said "no unwanted sexual comments").
golden_sexual = golden[golden["sexual"].notna()].copy()
golden_sexual["sexual"] = golden_sexual["sexual"].astype(int)

print(f"\nRows with a sexual label (our evaluation set): {len(golden_sexual)}")
print(f"  sexual=1 (contains unwanted sexual comments): {golden_sexual['sexual'].sum()}")
print(f"  sexual=0 (does not):                          {(golden_sexual['sexual'] == 0).sum()}")
print(f"\nSample:")
golden_sexual[["Review", "sexual"]].head(5)


Total rows in dataset:  56056
Rows in golden set:     331

How many have each label filled in:
  racism labeled:   329
  bullying labeled: 330
  sexual labeled:   330

Rows with a sexual label (our evaluation set): 330
  sexual=1 (contains unwanted sexual comments): 16
  sexual=0 (does not):                          314

Sample:


,Review,sexual
2,Get rid of micro transactions or i will find a new app to use. Why should i have to pay for that it’s so stupid,0
6,This is good but most of my messages never show up. This is very crapy and needs to be fixed,0
8,I was really enjoying this app. This brought me out of the box. I’m an extremely shy person and this gave me somewhere to talk to nice people. I just got kicked of bc I’m 16 not “18” and I think that this change it kind of stupid bc yeah it’s for protection but like someone else said all you hav...,0
13,It won’t lemme go live or anything like I think you fixed it for everyone but me and now it says I’m banned for no reason I didn’t even do anything,0
15,No real ppl all fake or no reply,0


---
## Step 3: Write the LLM Prompt

This is the part you'll experiment with! We're going to ask an LLM to read each review and decide whether it contains unwanted sexual content.

The prompt below is a starting point — it's intentionally simple. After you see the precision/recall results, you can come back and tweak the prompt to try to improve the scores.

**Things to experiment with:**
- How specific is your definition of "sexual content"?
- Do you ask for just a yes/no, or ask the model to reason first?
- Do you give examples?
- Do you set a strict or lenient threshold?

We're using [OpenRouter](https://openrouter.ai/) so you can swap in different models without changing the code structure.


In [13]:
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
MODEL = "openai/gpt-5.4-mini"  # ✏️ Change this to try different models

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Using model: {MODEL}")
print(f"API key loaded: {'yes' if OPENROUTER_API_KEY else 'NO - check your .env file'}")


Using model: openai/gpt-5.4-mini
API key loaded: yes


### Structured Outputs

Instead of asking the model to reply with "YES" or "NO" and trying to parse that text, we use **structured outputs** — we give the model a schema and it fills it in like a form.

We define two fields:
- `is_sexual` — a boolean (`True`/`False`)
- `reason` — a short string explaining the decision

The schema is passed directly to the API, so the model is *constrained* to return exactly this shape — not just asked nicely.


In [14]:
class ModerationResult(BaseModel):
    is_sexual: bool  # True if the review contains unwanted sexual comments
    reason: str      # Short explanation of why the model made this decision


### The Prompt

This is the part you'll experiment with! Write a system prompt and a user prompt that together tell the model what to look for.

**Things to try:**
- How specific is your definition of "unwanted sexual comments"?
- Do you ask the model to reason before deciding?
- Do you give examples of what counts (or doesn't count)?


In [15]:
# ✏️ Edit these prompts to experiment!

SYSTEM_PROMPT = """You are a content moderation assistant.
Your job is to detect whether app store reviews contain unwanted sexual comments.
Respond in JSON with two fields: "is_sexual" (boolean) and "reason" (string). 
Only return true if the comment is a compalint about sexual content that is unwanted by the reviewer, not just general discussion of sexual topics."""

USER_PROMPT = """Does the following app store review contain unwanted sexual comments?

Review: {review_text}"""


In [16]:
import json

def classify_sexual_content(review_text: str) -> ModerationResult:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(review_text=review_text)},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ModerationResult",
                "strict": True,
                "schema": ModerationResult.model_json_schema(),
            },
        },
        max_tokens=500,
        temperature=0,
    )
    data = json.loads(response.choices[0].message.content)
    return ModerationResult(**data)


### Sanity Check

Before running on all 331 reviews, test the function on one review to make sure it works.


In [17]:
test_review = "Great app for meeting people!"
result = classify_sexual_content(test_review)

print(f"Review:    {test_review}")
print(f"is_sexual: {result.is_sexual}")
print(f"reason:    {result.reason}")


Review:    Great app for meeting people!
is_sexual: False
reason:    The review is a general positive comment about meeting people and does not mention sexual content or an unwanted sexual complaint.


---
## Step 4: Run the Classifier on the Golden Set

Now we run `classify_sexual_content()` on every review in `golden_sexual` and store the results in a new column called `sexual_llm`. You'll see a progress bar as it runs.

> **Note:** We're only running this on rows that were actually labeled for sexual content — that's all we need for evaluation. Running it on all 56k rows would cost money and take a long time, and we don't have ground truth labels for those rows anyway.


In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def _run(idx_and_review):
    idx, review_text = idx_and_review
    for attempt in range(3):
        try:
            result = classify_sexual_content(str(review_text))
            return idx, result.is_sexual, result.reason
        except Exception:
            if attempt == 2:
                raise


Now run it on every review. This sends all the API calls in parallel and shows a progress bar as results come in.


In [19]:
reviews = list(golden_sexual["Review"].items())
sexual_llm, reasons = {}, {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(_run, item): item for item in reviews}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying reviews"):
        idx, is_sexual, reason = future.result()
        sexual_llm[idx] = int(is_sexual)
        reasons[idx] = reason

golden_sexual["sexual_llm"] = pd.Series(sexual_llm)
golden_sexual["reason"] = pd.Series(reasons)

print(f"LLM flagged as sexual: {golden_sexual['sexual_llm'].sum()}")
print(f"LLM said not sexual:   {(golden_sexual['sexual_llm'] == 0).sum()}")
golden_sexual[["Review", "sexual", "sexual_llm", "reason"]].head(10)


Classifying reviews:   0%|          | 0/330 [00:00<?, ?it/s]

LLM flagged as sexual: 34
LLM said not sexual:   296


,Review,sexual,sexual_llm,reason
2,Get rid of micro transactions or i will find a new app to use. Why should i have to pay for that it’s so stupid,0,0,"The review complains about microtransactions and pricing, with no sexual content or unwanted sexual comments."
6,This is good but most of my messages never show up. This is very crapy and needs to be fixed,0,0,The review complains about message delivery and app quality; it does not mention sexual content or unwanted sexual comments.
8,I was really enjoying this app. This brought me out of the box. I’m an extremely shy person and this gave me somewhere to talk to nice people. I just got kicked of bc I’m 16 not “18” and I think that this change it kind of stupid bc yeah it’s for protection but like someone else said all you hav...,0,0,"The review discusses an age restriction and dissatisfaction with being removed from the app, but it does not contain unwanted sexual comments or complaints about sexual content."
13,It won’t lemme go live or anything like I think you fixed it for everyone but me and now it says I’m banned for no reason I didn’t even do anything,0,0,"The review complains about app access and an apparent ban, with no sexual content or unwanted sexual comments."
15,No real ppl all fake or no reply,0,0,"The review complains about fake accounts or no responses, and does not mention sexual content or unwanted sexual comments."
17,Can’t join live and can’t scroll through profile and tap on them. Please fix ASAP. Everyone’s having this problem,0,0,The review discusses app functionality issues and does not mention sexual content or unwanted sexual comments.
19,Descent,0,0,"The review is only a single word, ""Descent,"" and does not contain any sexual content or complaint about unwanted sexual content."
22,Can't even get the app to open...there's no way the app can be any good if they can't even get it to open up without issues,0,0,The review complains about the app not opening and mentions no sexual content or unwanted sexual comments.
23,Y,0,0,The review contains only the single letter 'Y' and does not mention any sexual content or complaint about it.
24,Haven't met a single person on this app yet...,0,0,The review does not mention sexual content or unwanted sexual comments; it only says the user hasn’t met anyone on the app yet.


---
## Step 5: Calculate Precision and Recall

Now we compare the LLM's predictions (`sexual_llm`) against the human labels (`sexual`) to compute precision and recall.

First, let's understand the four possible outcomes for each review:

| | Human says: Sexual | Human says: Not Sexual |
|---|---|---|
| **LLM says: Sexual** | ✅ True Positive (TP) | ❌ False Positive (FP) |
| **LLM says: Not Sexual** | ❌ False Negative (FN) | ✅ True Negative (TN) |

- **True Positives (TP):** Correctly caught harmful content
- **False Positives (FP):** Flagged innocent reviews (false alarms)  
- **False Negatives (FN):** Missed actual harmful content
- **True Negatives (TN):** Correctly passed innocent reviews

Then:
- **Precision** = TP / (TP + FP) — "How accurate are the flags?"
- **Recall** = TP / (TP + FN) — "How much harmful content did we catch?"


In [20]:
human   = golden_sexual["sexual"]      # ground truth (0 or 1)
llm     = golden_sexual["sexual_llm"]  # LLM predictions (0 or 1)

# Count the four outcome types
tp = int(((human == 1) & (llm == 1)).sum())  # both said yes
fp = int(((human == 0) & (llm == 1)).sum())  # LLM said yes, human said no
fn = int(((human == 1) & (llm == 0)).sum())  # LLM said no, human said yes
tn = int(((human == 0) & (llm == 0)).sum())  # both said no

print("Confusion Matrix:")
print(f"  True Positives  (TP): {tp:>4}  — correctly flagged as unwanted sexual comments")
print(f"  False Positives (FP): {fp:>4}  — incorrectly flagged (false alarms)")
print(f"  False Negatives (FN): {fn:>4}  — missed actual unwanted sexual comments")
print(f"  True Negatives  (TN): {tn:>4}  — correctly passed as clean")

# Precision: of all the things we flagged, how many were actually unwanted sexual comments?
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

# Recall: of all the actual unwanted sexual comments, how many did we catch?
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# F1: harmonic mean of precision and recall — a single balanced score
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n{'='*40}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print(f"{'='*40}")
print(f"\nInterpretation:")
print(f"  - The LLM flagged {tp + fp} reviews as containing unwanted sexual comments")
print(f"  - Of those, {tp} matched the human label ({precision:.0%} precision)")
print(f"  - Out of {tp + fn} reviews humans labeled sexual=1, the LLM caught {tp} ({recall:.0%} recall)")


Confusion Matrix:
  True Positives  (TP):   16  — correctly flagged as unwanted sexual comments
  False Positives (FP):   18  — incorrectly flagged (false alarms)
  False Negatives (FN):    0  — missed actual unwanted sexual comments
  True Negatives  (TN):  296  — correctly passed as clean

  Precision: 47.06%
  Recall:    100.00%
  F1 Score:  64.00%

Interpretation:
  - The LLM flagged 34 reviews as containing unwanted sexual comments
  - Of those, 16 matched the human label (47% precision)
  - Out of 16 reviews humans labeled sexual=1, the LLM caught 16 (100% recall)


---
## Dig Deeper: Inspect Errors

It's useful to actually *read* the reviews that the model got wrong. Looking at false positives and false negatives often gives you intuition for how to improve the prompt.


In [23]:
pd.set_option("display.max_colwidth", None)

# False Positives: LLM flagged it, but humans said it was fine
false_positives = golden_sexual[(human == 0) & (llm == 1)][["Review", "sexual", "sexual_llm", "reason"]]
print(f"False Positives ({len(false_positives)} reviews the LLM incorrectly flagged):")
false_positives


False Positives (18 reviews the LLM incorrectly flagged):


,Review,sexual,sexual_llm,reason
72,Deberían de poner todo los personas que son gay activo o pasó is con fotos y sus datos generales como q relación prefieren su dirección y teléfono y ubicacione,0,1,"The review asks for profiles of gay people with photos, personal data, relationship preferences, address, phone number, and location, which is a sexualized/privacy-related request focused on unwanted sexual content."
74,"The idea of Omegle is pretty amusing. And seems fun but when it takes 30 minutes to find a person to talk to that isn't an underaged teenager looking to talk about ***, it seems pretty pointless.",0,1,"The review complains about encountering underaged users looking to talk about sexual content, which is an unwanted sexual comment."
76,"The app kicked me out & asked for my phone number to see if I was a real person. I didn't trust it so I plugged in a ""text app"" number instead & it's still not letting me back in. Also full of fake profiles, spam, tranynns, & gays when I strictly put my settings on women only. My main concern is that the app won't allow me back in though. I want to take my pictures down & never use this app again.",0,1,"The review includes unwanted sexual/relationship-related content complaints, specifically about being shown profiles that the reviewer did not want, including a derogatory mention of sexual/gender-related categories. It is a complaint about unwanted sexual content in the app."
80,The concept of the app is good but i have all my settings set to not receiving notifications when i get messages or things and i still get the notifications. It's just not for me. And i have only been contacted by creeps so i think i will delete this app.,0,1,"The reviewer mentions being contacted by ""creeps,"" which implies unwanted sexual or sexually suggestive attention on the app and is a complaint about that experience."
198,I loved this app! It WAS soo fun But now it’s only 18+ what’s with that? It was soo cool to talk to kids my age from different states and make new friends. While i did get some no so nice people it was fine because i meet so many amazing people. Why make it 18+? Your basically making it tinder with the option to video chat them. Please fix this.,0,1,"The reviewer complains that the app was changed to 18+ and says it is basically being made into Tinder with video chat, indicating unwanted sexualized content or adult dating-related use."
204,"This app has taken a turn for the worst. Since adding the dating/relationship option, this app has turned into a trashy meet and greet place, filled with drug, prostitution and disrespectful requests. Compliments turn into insults and none of your preferences are respected. More out of the country requests than local. People steal your pics and make fake profiles portraying u as the worse person ever. The racism exhibited on this app is insane and ridiculous. I'd say just log onto Facebook or IG as this app is horrible. Soddom and Gomorrah..smh",0,1,"The reviewer complains that the app has become filled with sexual and prostitution-related unwanted behavior, specifically mentioning prostitution and disrespectful sexual requests."
264,"Are you stupid? An app for ""making friends"" aka dating app for MINORS. Innocent ignorant children. Every kid thinks they're smart and won't fall into a trap, until they do and it's too late. This is a breeding ground for pedophiles are you kidding? Marketing an app to MINORS for meeting and possibly dating each other..that's just giving the pedos easy access come on..look for love in school or your town please..please. It's so much easier, healthier, and SAFE. Please stop, please be safe, this is a breeding ground for predators.",0,1,"The review complains about an app for minors being a breeding ground for pedophiles/predators and concerns about sexual exploitation of children, which is unwanted sexual content."
280,"This app is absolutely terrible! You will never make any meaningful friends, or find someone w

In [22]:
# False Negatives: Humans flagged it, but LLM missed it
false_negatives = golden_sexual[(human == 1) & (llm == 0)][["Review", "sexual", "sexual_llm", "reason"]]
print(f"False Negatives ({len(false_negatives)} reviews the LLM missed):")
false_negatives


False Negatives (0 reviews the LLM missed):


,Review,sexual,sexual_llm,reason
